# Hyperparameter Tuning for BiLSTM and Char-CNN Models

This notebook automates the hyperparameter tuning process for the BiLSTM (teacher) and Char-CNN (student) models. It evaluates the impact of key hyperparameters on model performance and visualizes the results.

In [ ]:
import itertools
import logging
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from train_models import Model1_BiLSTM, Model2_CharCNN, train_student_epoch, evaluate_student
from train_models import collate_pair_sequences, collate_word_profiles, PairSequenceDataset, WordProfileDataset
from torch.utils.data import DataLoader
import torch
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score

# Define paths and constants
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Logger setup
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("tuning")

# Load the dataset
from train_models import resolve_data_path, validate_required_columns
data_path = resolve_data_path()
df = pd.read_excel(data_path)

# Validate required columns
validate_required_columns(df)

# Preprocess the dataset
from train_models import CharTokenizer, build_tokenizer_from_pairs

# Combine "correct" and "incorrect" columns into a list of pairs
pairs = list(zip(df["correct"], df["incorrect"]))

# Build the tokenizer using the list of pairs
tokenizer = build_tokenizer_from_pairs(pairs)

# Encode the error types
label_encoder = LabelEncoder()
df["error_type_encoded"] = label_encoder.fit_transform(df["error_type"])

# Split the dataset into training and validation sets
from sklearn.model_selection import train_test_split
train_pairs, val_pairs, train_labels, val_labels = train_test_split(
    list(zip(df["correct"], df["incorrect"])),
    df["error_type_encoded"],
    test_size=0.2,
    random_state=42,
)

# Create DataLoader objects for BiLSTM
train_dataset = PairSequenceDataset(train_pairs, train_labels, tokenizer)
val_dataset = PairSequenceDataset(val_pairs, val_labels, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_pair_sequences,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_pair_sequences,
)

# Create teacher embeddings for Char-CNN
from train_models import Model1_BiLSTM
teacher_model = Model1_BiLSTM(
    vocab_size=tokenizer.vocab_size,
    embed_dim=64,
    hidden_dim=128,
    num_layers=2,
    dropout=0.3,
    num_classes=len(label_encoder.classes_),
).to(DEVICE)
teacher_model.eval()

teacher_embeddings = []
with torch.no_grad():
    for x_batch, _ in train_loader:
        x_batch = x_batch.to(DEVICE)
        _, embeddings = teacher_model(x_batch)
        teacher_embeddings.append(embeddings.cpu())
teacher_embeddings = torch.cat(teacher_embeddings, dim=0)

# Create DataLoader objects for Char-CNN
train_word_dataset = WordProfileDataset(df["correct"], teacher_embeddings, tokenizer)
train_word_loader = DataLoader(
    train_word_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_word_profiles,
)
val_word_loader = DataLoader(
    train_word_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_word_profiles,
)


# Define hyperparameter grids
bilstm_params = {
    "embed_dim": [32, 64],
    "hidden_dim": [64, 128],
    "num_layers": [1, 2],
    "dropout": [0.3, 0.5],
}

charcnn_params = {
    "embed_dim": [32, 64],
    "num_filters": [64, 128],
    "kernel_sizes": [(2, 3, 4), (3, 4, 5)],
    "dropout": [0.3, 0.5],
}

training_params = {
    "batch_size": [32, 64],
    "learning_rate": [1e-3, 5e-4],
    "num_epochs": [10],
}

# Function to train and evaluate BiLSTM
def train_and_evaluate_bilstm(params, train_loader, val_loader):
    model = Model1_BiLSTM(
        vocab_size=tokenizer.vocab_size,
        embed_dim=params["embed_dim"],
        hidden_dim=params["hidden_dim"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
        num_classes=num_classes,
    ).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"])
    for epoch in range(params["num_epochs"]):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            logits, embeddings = model(x_batch)
            loss = F.cross_entropy(logits, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluate on validation set
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            logits, _ = model(x_batch)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    logger.info("BiLSTM Accuracy: %.4f", accuracy)
    return accuracy

# Function to train and evaluate Char-CNN
def train_and_evaluate_charcnn(params, train_loader, val_loader, teacher_embeddings):
    model = Model2_CharCNN(
        vocab_size=tokenizer.vocab_size,
        embed_dim=params["embed_dim"],
        num_filters=params["num_filters"],
        kernel_sizes=params["kernel_sizes"],
        dropout=params["dropout"],
    ).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"])
    for epoch in range(params["num_epochs"]):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            predicted_embeddings = model(x_batch)
            loss = F.mse_loss(predicted_embeddings, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluate on validation set
    model.eval()
    cosine_similarities = []
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            predicted_embeddings = model(x_batch)
            cosine_sim = F.cosine_similarity(predicted_embeddings, y_batch, dim=1)
            cosine_similarities.extend(cosine_sim.cpu().numpy())

    avg_cosine_similarity = sum(cosine_similarities) / len(cosine_similarities)
    logger.info("Char-CNN Cosine Similarity: %.4f", avg_cosine_similarity)
    return avg_cosine_similarity


In [ ]:
# Run experiments
results = []
for bilstm_config in itertools.product(*bilstm_params.values()):
    bilstm_config = dict(zip(bilstm_params.keys(), bilstm_config))
    for charcnn_config in itertools.product(*charcnn_params.values()):
        charcnn_config = dict(zip(charcnn_params.keys(), charcnn_config))
        for training_config in itertools.product(*training_params.values()):
            training_config = dict(zip(training_params.keys(), training_config))

            logger.info("Testing configuration: BiLSTM %s, Char-CNN %s, Training %s", bilstm_config, charcnn_config, training_config)

            # Train and evaluate BiLSTM
            bilstm_accuracy = train_and_evaluate_bilstm(bilstm_config, train_loader, val_loader)

            # Train and evaluate Char-CNN
            charcnn_cosine_similarity = train_and_evaluate_charcnn(charcnn_config, train_word_loader, val_word_loader, teacher_embeddings)

            # Log results
            results.append({
                "bilstm_config": bilstm_config,
                "charcnn_config": charcnn_config,
                "training_config": training_config,
                "bilstm_accuracy": bilstm_accuracy,
                "charcnn_cosine_similarity": charcnn_cosine_similarity,
            })

# Save results
with open(RESULTS_DIR / "tuning_results.json", "w") as f:
    json.dump(results, f, indent=4)
logger.info("Tuning completed. Results saved to %s", RESULTS_DIR / "tuning_results.json")


In [ ]:
# Function to plot results
def plot_results(results, metric, model_name):
    """
    Plot the results of hyperparameter tuning.
    
    Args:
        results (list): List of dictionaries containing experiment results.
        metric (str): The metric to plot (e.g., "bilstm_accuracy", "charcnn_cosine_similarity").
        model_name (str): Name of the model (e.g., "BiLSTM", "Char-CNN").
    """
    df = pd.DataFrame(results)
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x="bilstm_config", y=metric)
    plt.title(f"{model_name} Hyperparameter Tuning: {metric}")
    plt.xlabel("Hyperparameter")
    plt.ylabel(metric)
    plt.show()

In [ ]:
# Visualize BiLSTM accuracy
plot_results(results, metric="bilstm_accuracy", model_name="BiLSTM")

# Visualize Char-CNN cosine similarity
plot_results(results, metric="charcnn_cosine_similarity", model_name="Char-CNN")